# I-JEPA pre-training on a Kaggle GPU

Trains one run of `jepa/config.py` on a GPU and writes its outputs in the
same layout as the local project, so the analysis scripts can read them
directly.

**Before running**, in the notebook settings (right-hand panel):
- *Accelerator*: **GPU T4 x1** (or P100);
- *Internet*: **on** (needs a phone-verified Kaggle account), to clone the
  repo and download the dataset.

**To train without keeping the tab open**: *Save Version* -> *Save & Run All
(Commit)*. The outputs then appear in the *Output* tab of that version.

Outputs, in `/kaggle/working/`:
- `checkpoints/<run>/jepa.weights.h5` (saved after every epoch);
- `logs/<run>.log` (read by `result_analysis/plot_loss.py`);
- `<run>.zip` with both, to unzip at the project root.

## 1. Parameters

In [1]:
RUN = "cifar10_d256"   # a preset of jepa/config.py: cifar10_d128, cifar10_d256, stl10_d128, stl10_d256
BRANCH = "main"
EPOCHS = None          # None = config.epochs (30)

REPO_URL = "https://github.com/maxzaragoza2a/i-jepa-from-scratch.git"
REPO_DIR = "/tmp/i-jepa-from-scratch"   # outside /kaggle/working: the code is not an output
WORK = "/kaggle/working"

## 2. Check the GPU and the libraries

Kaggle's own TensorFlow is kept: reinstalling the version pinned in
`requirements.txt` could break the CUDA setup. The code only needs Keras 3.

In [2]:
!nvidia-smi

import tensorflow as tf
import keras

print("tensorflow", tf.__version__, "| keras", keras.__version__)
gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)
assert gpus, "No GPU: set Accelerator to a GPU in the notebook settings."
assert int(keras.__version__.split(".")[0]) >= 3, "The code needs Keras 3."

Fri Oct 23 13:03:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 3. Get the code

In [3]:
import os

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {REPO_DIR}
!git -C {REPO_DIR} log --oneline -1

Cloning into '/tmp/i-jepa-from-scratch'...
remote: Enumerating objects: 44, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (42/42), done.
remote: Total 44 (delta 0), reused 23 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (44/44), 256.48 KiB | 9.50 MiB/s, done.
abad359 (grafted, HEAD -> main, origin/main, origin/HEAD) Track the Kaggle GPU training in the TODO


## 4. Dependencies

Only STL-10 needs `tensorflow_datasets` (often already installed on Kaggle).
`importlib_resources` is a dependency tfds needs but does not declare.

In [4]:
try:
    import tensorflow_datasets  # noqa: F401
except ImportError:
    %pip install -q tensorflow-datasets
%pip install -q importlib_resources

ERROR:absl:Detected mismatched Protobuf Gencode/Runtime major versions when loading tensorflow_metadata/proto/v0/anomalies.proto: gencode 6.31.1 runtime 5.29.5. Same major version is required. See Protobuf version guarantees at https://protobuf.dev/support/cross-version-runtime-guarantee.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/__init__.py", line 79, in <module>
    from tensorflow_datasets import rlds  # pylint: disable=g-bad-import-order
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/rlds/__init__.py", line 21, in <module>
    from tensorflow_datasets.rlds import envlogger_reader
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/rlds/envlogger_reader.py", line 21, in <module>
    from tensorflow_datasets.core.utils.lazy_imports_utils import tree
  File "/usr/local/lib/python3.13/dist-packages/tensorflow_datasets/core/__init__.py", line 21, in 

Note: you may need to restart the kernel to use updated packages.


## 5. Smoke test

20 steps, written to `/tmp`. Checks that the run works on this setup and
gives a rough time per step before the full run.

In [5]:
%%time
!cd {REPO_DIR} && python -u -m jepa.train --run {RUN} --max-steps 20 --out /tmp/smoke

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 1271s 7us/step
I0000 00:00:1792761943.951195      84 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1792761943.956556      84 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
parameters: 3.63M trainable, 3.21M frozen (target encoder, positions)
30 epochs x 390 steps, 20 steps in all, 2 of warmup
epoch   1/30  loss 0.6027  lr 0.00e+00  ema 1.0000  27s
CPU times: user 22.1 s, sys: 5.53 s, total: 27.7 s
Wall time: 21min 58s


## 6. Full training

The log goes to the notebook and to `logs/<run>.log` at the same time.
The weights are saved after every epoch.

In [6]:
epochs_arg = f"--epochs {EPOCHS}" if EPOCHS else ""
os.makedirs(f"{WORK}/logs", exist_ok=True)

!cd {REPO_DIR} && python -u -m jepa.train --run {RUN} {epochs_arg} --out {WORK}/checkpoints/{RUN} 2>&1 | tee {WORK}/logs/{RUN}.log

I0000 00:00:1792761994.352975     275 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1792761994.355878     275 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
parameters: 3.63M trainable, 3.21M frozen (target encoder, positions)
30 epochs x 390 steps, 11700 steps in all, 1170 of warmup
epoch   1/30  loss 0.3346  lr 3.33e-04  ema 0.9960  484s
epoch   2/30  loss 0.0923  lr 6.67e-04  ema 0.9960  479s
epoch   3/30  loss 0.1143  lr 1.00e-03  ema 0.9961  479s
epoch   4/30  loss 0.1531  lr 9.97e-04  ema 0.9962  486s
epoch   5/30  loss 0.1653  lr 9.87e-04  ema 0.9963  470s
epoch   6/30  loss 0.1917  lr 9.70e-04  ema 0.9964  473s
epoch   7/30  loss 0.2196  lr 9.47e-04  ema 0.9965  471s
epoch   8/30  loss 0.2497  lr 9

## 7. Package the outputs

In [7]:
!cd {WORK} && rm -f {RUN}.zip && zip -r {RUN}.zip checkpoints/{RUN} logs/{RUN}.log
!ls -lh {WORK}/{RUN}.zip

  adding: checkpoints/cifar10_d256/ (stored 0%)
  adding: checkpoints/cifar10_d256/jepa.weights.h5 (deflated 9%)
  adding: logs/cifar10_d256.log (deflated 78%)
-rw-r--r-- 1 root root 25M Oct 23 17:23 /kaggle/working/cifar10_d256.zip


## 8. Back on the local machine

Download `<run>.zip` from the *Output* tab and unzip it at the project root:
it gives `checkpoints/<run>/jepa.weights.h5` and `logs/<run>.log`. Then:

```bash
python result_analysis/plot_loss.py --run cifar10_d256
python result_analysis/check_collapse.py --run cifar10_d256
python result_analysis/predicted_fill.py --run cifar10_d256
```